In [138]:
# Export model, Transformer and encoder
import pickle as pkl
pkl.dump(model,open("model.pkl","wb"))
pkl.dump(encoder,open("encoder.pkl",'wb'))

pkl.dump(featureEncoder,open('feature11.pkl','wb'))

In [1]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,PowerTransformer,StandardScaler,LabelEncoder
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, accuracy_score,f1_score,fbeta_score
import pandas as pd

In [2]:
weatherData=pd.read_csv("weather-dataset.csv")

In [3]:
weatherDa

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain
...,...,...,...,...,...,...
1456,2015-12-27,8.6,4.4,1.7,2.9,rain
1457,2015-12-28,1.5,5.0,1.7,1.3,rain
1458,2015-12-29,0.0,7.2,0.6,2.6,fog
1459,2015-12-30,0.0,5.6,-1.0,3.4,sun


In [4]:
# extract Year, Month and day from date

In [5]:
weatherData['date']=pd.to_datetime(weatherData['date'])

In [6]:
# getYear
weatherData['Year']=weatherData['date'].dt.year

# getMonths
weatherData['Month']=weatherData['date'].dt.month_name()



In [7]:
# get day
weatherData['Day']=weatherData['date'].dt.day


In [8]:
weatherData.describe

<bound method NDFrame.describe of            date  precipitation  temp_max  temp_min  wind  weather  Year  \
0    2012-01-01            0.0      12.8       5.0   4.7  drizzle  2012   
1    2012-01-02           10.9      10.6       2.8   4.5     rain  2012   
2    2012-01-03            0.8      11.7       7.2   2.3     rain  2012   
3    2012-01-04           20.3      12.2       5.6   4.7     rain  2012   
4    2012-01-05            1.3       8.9       2.8   6.1     rain  2012   
...         ...            ...       ...       ...   ...      ...   ...   
1456 2015-12-27            8.6       4.4       1.7   2.9     rain  2015   
1457 2015-12-28            1.5       5.0       1.7   1.3     rain  2015   
1458 2015-12-29            0.0       7.2       0.6   2.6      fog  2015   
1459 2015-12-30            0.0       5.6      -1.0   3.4      sun  2015   
1460 2015-12-31            0.0       5.6      -2.1   3.5      sun  2015   

         Month  Day  
0      January    1  
1      January    2  

In [119]:
weatherData.to_csv("test_dataset.csv")


In [10]:
# check for null values
weatherData.isnull().sum()

date             0
precipitation    0
temp_max         0
temp_min         0
wind             0
weather          0
Year             0
Month            0
Day              0
dtype: int64

In [11]:
# extract columns(num and text cols)
numCols=[cols for cols in weatherData.columns if weatherData[cols].dtype!="object"]
textCols=[cols for cols in weatherData.columns if weatherData[cols].dtype in ["int64","float64"]]

In [12]:
numCols


['date', 'precipitation', 'temp_max', 'temp_min', 'wind', 'Year', 'Day']

In [13]:
numCols

['date', 'precipitation', 'temp_max', 'temp_min', 'wind', 'Year', 'Day']

In [14]:
weatherData.dtypes

date             datetime64[ns]
precipitation           float64
temp_max                float64
temp_min                float64
wind                    float64
weather                  object
Year                      int32
Month                    object
Day                       int32
dtype: object

In [15]:
excludeColumns=['date','Year','Day']

In [16]:
for skew in numCols:
    if skew not in excludeColumns:
        plt.figure(figsize=(20,5))
        sk = weatherData[skew].skew()
        print(sk)

3.505643716998874
0.2809299923916159
-0.2494585516131789
0.8916675191285183


<Figure size 2000x500 with 0 Axes>

<Figure size 2000x500 with 0 Axes>

<Figure size 2000x500 with 0 Axes>

<Figure size 2000x500 with 0 Axes>

In [17]:
numCols

['date', 'precipitation', 'temp_max', 'temp_min', 'wind', 'Year', 'Day']

In [122]:
# split data
x=weatherData.drop(columns=['date','weather'])
y=weatherData['weather']
xTrain,xTest,yTrain,yTest=train_test_split(x,y,test_size=0.3,random_state=36)

In [123]:
pipelineText=Pipeline(
    [
        ("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ]
)
featureEncoder=ColumnTransformer(
    [
        
         ("encodeText",pipelineText,textCols)
        
    ]
)

In [124]:
# process xTrain
xTrain=featureEncoder.fit_transform(xTrain)

In [125]:
xTest=featureEncoder.transform(xTest)

In [109]:
xtt=pd.DataFrame(xTrain)

In [110]:
yt=pd.DataFrame(yTrain)

In [126]:
# encode category
encoder=LabelEncoder()
yTrain=encoder.fit_transform(yTrain)

In [127]:
yTest=encoder.transform(yTest)

In [113]:
conc=pd.concat([xtt.reset_index(drop=True),yt.reset_index(drop=True)],axis=1)


# withFeatures=pd.DataFrame(conc,columns=Transformer.get_feature_names_out())

In [128]:
model=XGBClassifier(

    learning_rate=0.1,
    reg_alpha=0.1,
    reg_lambda=0.1,
    n_estimators=700
)
model.fit(xTrain,yTrain)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [129]:
prediction=savedPipeline.named_steps['model'].predict(xTest)

In [130]:
classReport=classification_report(yTest,prediction,target_names=encoder.classes_,output_dict=True)
classReport

{'drizzle': {'precision': 0.14285714285714285,
  'recall': 0.058823529411764705,
  'f1-score': 0.08333333333333333,
  'support': 17.0},
 'fog': {'precision': 0.07692307692307693,
  'recall': 0.03225806451612903,
  'f1-score': 0.045454545454545456,
  'support': 31.0},
 'rain': {'precision': 0.9263157894736842,
  'recall': 0.9312169312169312,
  'f1-score': 0.9287598944591029,
  'support': 189.0},
 'snow': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 8.0},
 'sun': {'precision': 0.7665198237885462,
  'recall': 0.8969072164948454,
  'f1-score': 0.8266033254156769,
  'support': 194.0},
 'accuracy': 0.8018223234624146,
 'macro avg': {'precision': 0.38252316660849,
  'recall': 0.3838411483279341,
  'f1-score': 0.3768302197325317,
  'support': 439.0},
 'weighted avg': {'precision': 0.748500493937793,
  'recall': 0.8018223234624146,
  'f1-score': 0.7715772727999302,
  'support': 439.0}}

In [131]:
reportFrame=pd.DataFrame(classReport).transpose()
reportFrame

,precision,recall,f1-score,support
drizzle,0.142857,0.058824,0.083333,17.000000
fog,0.076923,0.032258,0.045455,31.000000
rain,0.926316,0.931217,0.928760,189.000000
snow,0.000000,0.000000,0.000000,8.000000
sun,0.766520,0.896907,0.826603,194.000000
accuracy,0.801822,0.801822,0.801822,0.801822
macro avg,0.382523,0.383841,0.376830,439.000000
weighted avg,0.748500,0.801822,0.771577,439.000000
